<a href="https://colab.research.google.com/github/saradom11/Simulaci-n-I/blob/main/Sistema_de_l%C3%ADneas_de_espera_con_un_servidor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Sistemas líneas de espera**

Considere una estación de servicio a la cual los clientes llegan de acuerdo con un proceso Poisson no homogéneo con función de intensidad $\lambda(t)$, $\lambda \geq 0$. Hay un único servidor, y al llegar un cliente pasa a servicio si el servidor está libre en ese momento, o bien se une a la fila de espera si está ocupado. Cuando el servidor termina de dar servicio a un cliente, se ocupa del cliente que ha estado esperando más tiempo (la disciplina "primero en llegar, primero en atender") si hay clientes esperando, o bien, si no los hay, permanece libre hasta la llegada del siguiente cliente. El tiempo que tarda la atención a un cliente es una variable aleatoria (independiente de los demás tiempos de servicio y del proceso de llegada) con distribución de probabilidad $G$. Además, hay un tiempo fijo $T$ después del cual no se permite que otras llegadas entren al sistema, aunque el servidor atiende a todos los que ya estén dentro del sistema en el instante $T$.

Suponga que estamos interesados en simular el sistema anterior para determinar cantidades tales como: \
(a) el tiempo promedio que pasa un cliente dentro del sistema, y \
(b) el tiempo promedio posterior a $T$ cuando sale el último cliente; es decir, el tiempo promedio en que el servidor puede ir a casa.



Para simular el sistema anterior utilizamos las siguientes variables: \
$\text{Variable de tiempo:} \qquad t$ \
$\text{Variables de conteo:} \qquad N_A: \text{El número de llegadas hasta el instante t}$ \
$$N_D: \text{El número de salidas hasta el instante t}$$
$\text{Variable de estado de sistema:} \quad n:\text{El número de clientes en el sistema en el tiempo t}$



Como el instante natural para modificar las cantidades anteriores es cuando ocurre una llegada o una salida, las consideramos "eventos"; es decir, hay dos tipos de eventos: llegadas y salidas. La lista de eventos contiene el instante de la siguiente llegada y el instante de la salida del cliente que se encuentra en servicio. Es decir, la lista de eventos es:

$$\textbf{LE} = t_A, t_D$$

donde $t_A$ es la hora de la siguiente llegada y $t_D$ es la hora a la que concluye el servicio del cliente que se está atendiendo actualmente. Si no hay clientes en servicio, $t_D$ es igual a $\infty$.

Las variables de salida por registrar son $A(i)$, la hora de llegada del cliente $i$; $D(i)$, la hora de salida del cliente $i$, y $T_p$, el tiempo de salida del último cliente, posterior a $T$.

In [13]:
#Importamos las librerias
import numpy as np
import matplotlib.pyplot as plt
import random as r
import math

In [14]:
def simular_linea_de_espera(lambda_t, servicio_dist, T, lambda_max, semilla):
    #Parámetros ----- lambda_t :función λ(t)   (Intensidad de llegadas)
    #servicio_dist :función  (Generador de tiempos de servicio con distribución G)
    #T :float (Tiempo de cierre. Después de T ya no se aceptan llegadas)
    #lambda_max :float (Cota superior de λ(t))
    # La semilla nos ayuda para controlar los números aleatorios que genera la simulación

    #Inicializamos las variables
    np.random.seed(semilla)

    t = 0
    N_A = 0  #Número de llegadas
    N_D = 0  #Número de salidas
    n = 0    #Número de clientes

    #Generar T_0 (Proceso de Poisson no homogéneo)
    t_A = poisson_no_homogeneo(lambda_t, t, T, lambda_max)
    t_D = np.inf

    A = []  #Almacenar horas de llegada
    D = []  #Almacenar horas de salida

    while True:

        #Caso 1
        if t_A <= t_D and t_A <= T:
            t = t_A
            N_A += 1
            n += 1
            A.append(t)

            t_A = poisson_no_homogeneo(lambda_t, t, T, lambda_max)

            if n == 1:
                y = servicio_dist()
                t_D = t + y

        #Caso 2
        elif t_D < t_A and t_D <= T:
            t = t_D
            n -= 1
            N_D += 1
            D.append(t)

            if n == 0:
                t_D = np.inf
            else:
                y = servicio_dist()
                t_D = t + y

        #Caso 3
        elif min(t_A, t_D) > T and n > 0:
            t = t_D
            n -= 1
            N_D += 1
            D.append(t)

            if n == 0:
                break
            else:
                y = servicio_dist()
                t_D = t + y

        #Caso 4
        elif min(t_A, t_D) > T and n == 0:
            break

    T_p = D[-1] if D else 0  #Hora de salida del último cliente
    T_extra = max(T_p - T, 0)

    return A, D, T_p, T_extra

# Retorna ------- A :lista (Tiempos de llegada)
                # D :lista (Tiempos de salda)
                # Tp :float (Tiempo en que sale el último cliente)
                # T_extra :float (Tiempo trabajado después de T)

Para generalizar el código se pudso el *Proceso de Poisson No Homogéneo* en el código ya que, la idea es que la tasa de llegadas ($λ(t)$) cambie con el tiempo, por lo que no podemos usar directamente una exponencial con parámetro fijo.



**Pero en este caso** la tasa es constante $\frac {lambda\ t}{lambda\ max}= 1 \quad$ y

la condición se vuelve:  np.random.uniform() <= 1, la cual siempre se cumple.Por lo que, nunca se rechaza ningún candidato y el algoritmo se reduce a generar tiempos entre llegadas exponenciales con media: 1/4

que justamente es un *Proceso de Poisson Homogéneo*.

In [15]:
def poisson_no_homogeneo(lambda_t, t_actual, T, lambda_max):
    t = t_actual
    while t <= T:
        u = np.random.exponential(1 / lambda_max)
        t = t + u
        if t > T:
            return np.inf
        if np.random.uniform() <= lambda_t(t) / lambda_max:
            return t
    return np.inf

Utilizando los valores indicados de:
$$ μ = 6 \qquad y \qquad λ = 4$$

In [16]:
def lambda_t(t):  #Intensidad de llegadas λ(t)
    return 4

#Distribución de servicio
def dist_servicio():  #Tiempo de servicio ~ Exponencial(media=0.4)
    return np.random.exponential(1/6)

T = 20  #Tiempo de cierre

#Ejecutar simulación
A, D, Tp, T_extra = simular_linea_de_espera(lambda_t=lambda_t,servicio_dist=dist_servicio,T=T,lambda_max=4,semilla=42)

#Calculo de (a) y (b)
tiempo_en_sistema = np.array(D) - np.array(A)
W_q = np.mean(tiempo_en_sistema)

print("-----------Resultados------------")
print(f"(a) Tiempo promedio en sistema W_q = {W_q:.4f}")
print(f"(b) Tiempo extra después del cierre = {T_extra:.4f}")


-----------Resultados------------
(a) Tiempo promedio en sistema W_q = 0.4018
(b) Tiempo extra después del cierre = 0.5382
